# CUSTOMHyS en la práctica
CUSTOMHyS construye algoritmos de optimización combinando **operadores de búsqueda**, y una **hiperheurística** puede elegir esa combinación por usted. Este cuaderno sigue el orden en que suele usarse el framework: funciones de prueba, población, operadores, metaheurísticas, hiperheurísticas y, al final, un problema escrito desde cero (Rosenbrock).

Tutorial original de Jorge Mario Cruz-Duarte, basado en el [artículo de CUSTOMHyS en SoftwareX](https://doi.org/10.1016/j.softx.2020.100628).
[English](Tutorial_English.ipynb) · Español · [Português (Brasil)](Tutorial_Portuguese_BR.ipynb)

## 0. Preparación
Requiere Python 3.10–3.12. En la raíz del repositorio:

```shell
python -m pip install -e ".[examples]"
```

No hacen falta LaTeX ni TensorFlow (el aviso de TensorFlow al importar puede ignorarse). Todos los problemas son de **minimización**.

Las iteraciones, los individuos y las repeticiones son pocos para que el cuaderno corra rápido. Para resultados más fiables, auméntelos y use más semillas.

Si modifica el código del framework, reinicie el kernel: reimportar no recarga módulos ya cargados, ni siquiera con `pip install -e`. La celda siguiente imprime qué `operators.py` se está usando.

In [ ]:
import sys
import ast
import random
from importlib.resources import files
from pathlib import Path
from uuid import uuid4

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import customhys
from customhys import benchmark_func as bf
from customhys import hyperheuristic as hh
from customhys import metaheuristic as mh
from customhys import operators as op
from customhys import population as pp
from customhys import tools as tl

print("Python:", sys.executable)
print("operators.py:", op.__file__)

%matplotlib inline
plt.rcParams.update({"text.usetex": False, "figure.figsize": (7, 4)})
SEED = 42
NUM_AGENTS = 32
NUM_ITERATIONS = 60
NUM_REPLICAS = 3
HH_STEPS = 8
np.random.seed(SEED)
random.seed(SEED)
print("CUSTOMHyS", customhys.__version__)

labels = {
    "iteration": "Iteración",
    "step": "Paso de operador registrado",
    "fitness": "Valor objetivo",
    "best": "Mejor hasta ahora",
    "current": "Actual",
    "position": "Posición (dominio original)",
    "init": "Inicialización",
    "operator": "Operador",
    "parameters": "Parámetros",
    "selector": "Selector",
    "replica": "Réplica",
    "performance": "Desempeño (mediana + RIC)",
    "centroid": "Centroide (normalizado)",
    "radius": "Radio (normalizado)",
}

labels.update(
    {
        "snapshot": "Población",
        "record": "Aplicación de operador",
        "trajectory": "Cinco individuos y el mejor punto",
        "occupancy": "Dónde permaneció la población",
        "count": "Posiciones registradas",
        "optimum": "Mínimo conocido",
        "convergence": "Mejor valor objetivo",
        "init_title": "Puntos iniciales",
        "unique": "Puntos distintos",
        "edge": "Puntos en los límites (%)",
        "bins": "Celdas ocupadas (8 × 8)",
        "start": "Inicial",
        "end": "Final",
        "distance": "Distancia a (1, 1)",
        "search": "Búsqueda del algoritmo",
        "replay": "Algoritmo elegido",
        "initial": "Valor objetivo inicial",
    }
)


## 1. Funciones de prueba
El módulo `benchmark_func` reúne las funciones de prueba. Cada una se crea con su número de dimensiones y trae dominio y mínimo conocidos. Rastrigin tiene muchos mínimos locales y su mínimo global vale cero en el origen; `plot()` dibuja la superficie en dos dimensiones.

In [ ]:
print(len(bf.__all__), bf.__all__)
fun = bf.Rastrigin(2)
fun.plot(samples=40, resolution=100)


`get_function_value` evalúa un punto. Aquí arma la malla `X, Y, Z` para dibujar la superficie a mano. Reutilizaremos esa malla como fondo en los gráficos de la población.

In [ ]:
bounds = np.asarray(fun.get_search_range()).T
X, Y = np.meshgrid(*(np.linspace(low, high, 40) for low, high in bounds))
Z = np.array(
    [fun.get_function_value([x, y]) for x, y in zip(X.ravel(), Y.ravel(), strict=True)]
).reshape(X.shape)
fig = plt.figure()
ax = fig.add_subplot(projection="3d")
ax.plot_surface(X, Y, Z, cmap="viridis", alpha=0.8)
ax.set(xlabel="$x_1$", ylabel="$x_2$", zlabel=labels["fitness"])
plt.show()


El dominio, la escala, el desplazamiento y el ruido cambian con los métodos `set_*`. Usamos una instancia aparte de Ackley para esto. Con ruido, evaluar el mismo punto dos veces da valores distintos, así que los demás ejemplos usan funciones sin modificar.

In [ ]:
modified_fun = bf.Ackley1(2)
modified_fun.set_search_range([-1.0, -1.0], [1.0, 1.0])
modified_fun.set_offset_function(-2)
modified_fun.set_scale_function(2)
modified_fun.set_noise_level(0.1)
modified_fun.set_offset_domain([0.5, 0.5])
modified_fun.plot(samples=40)


## 2. Población e inicialización
`Population` guarda las posiciones, el valor de la función de cada individuo y los mejores registros (por individuo y global). Internamente las posiciones están en `[-1, 1]`; `get_positions()` las convierte al dominio del problema.

El orden habitual es: inicializar (`initialise_positions`), evaluar (`evaluate_fitness`) y actualizar los registros de población, mejor personal y mejor global (`update_positions`). Los operadores esperan que esos registros estén llenos. La tabla muestra la misma coordenada en ambas escalas.

In [ ]:
fun = bf.Rastrigin(2)
pop = pp.Population(fun.get_search_range(), num_agents=NUM_AGENTS)
pop.initialise_positions("lhs")
pop.evaluate_fitness(fun.get_function_value)
pop.update_positions("population", "all")
pop.update_positions("particular", "all")
pop.update_positions("global", "greedy")
display(
    pd.DataFrame(
        {
            "x1 [-1, 1]": pop.positions[:, 0],
            "x1 [-5.12, 5.12]": pop.get_positions()[:, 0],
            labels["fitness"]: pop.fitness,
        }
    ).head()
)
print(pop.get_state())


In [ ]:
def show_positions(population):
    positions = population.get_positions()
    fig, ax = plt.subplots()
    ax.contour(X, Y, Z, levels=15, cmap="viridis", alpha=0.5)
    ax.scatter(positions[:, 0], positions[:, 1], c="tab:red", s=20)
    ax.set(xlabel="$x_1$", ylabel="$x_2$")
    plt.show()


show_positions(pop)


### 2.1 Métodos de inicialización
`initialise_positions` acepta varios esquemas. Los gráficos siguientes muestran solo los puntos de partida (todavía no hay búsqueda): 64 individuos en un dominio cuadrado. Los parámetros de las distribuciones se refieren al intervalo unitario, antes de convertir al dominio.

| Método | Cómo reparte los puntos |
| --- | --- |
| `random` | Sorteo uniforme. Puede dejar huecos y formar aglomeraciones. |
| `vertex` | Vértices de cuadrados anidados; no es una malla cartesiana completa. |
| `lhs` | Hipercubo latino: un punto en cada una de las 64 franjas de cada eje. |
| `sobol`, `halton` | Secuencias de baja discrepancia que reparten los puntos con regularidad. `scramble=False` hace la muestra reproducible. |
| `beta` | Parámetros menores que 1 llevan los puntos a los bordes; mayores que 1, al centro. |
| `normal` | Concentra los puntos cerca del centro (media 0,5 y desviación 0,25 en el intervalo unitario). |
| `lognormal`, `exponential`, `rayleigh` | Distribuciones asimétricas. Los valores se recortan a `[0, 1]` antes de la conversión, así que los puntos se acumulan en los límites; se ve en la lognormal con estos parámetros. |

In [ ]:
INITIAL_AGENTS = 64
initialisation_methods = [
    ("random", "random", {}),
    ("vertex", "vertex", {}),
    ("lhs", "lhs", {}),
    ("sobol", "sobol", {"scramble": False}),
    ("halton", "halton", {"scramble": False}),
    ("beta (0.5, 0.5)", "beta", {"a": 0.5, "b": 0.5}),
    ("beta (2, 2)", "beta", {"a": 2.0, "b": 2.0}),
    ("normal (0.5, 0.25)", "normal", {"mean": 0.5, "std": 0.25}),
    ("lognormal (0, 0.5)", "lognormal", {"mean": 0.0, "sigma": 0.5}),
    ("exponential (0.5)", "exponential", {"scale": 0.5}),
    ("rayleigh (0.4)", "rayleigh", {"scale": 0.4}),
]
initial_samples = {}
for name, scheme, options in initialisation_methods:
    np.random.seed(SEED)
    sample_pop = pp.Population(([-5.0, -5.0], [5.0, 5.0]), num_agents=INITIAL_AGENTS)
    sample_pop.initialise_positions(scheme, **options)
    initial_samples[name] = sample_pop.get_positions().copy()

initialisation_figures = []
for group in (initialisation_methods[:5], initialisation_methods[5:]):
    fig, axes = plt.subplots(2, 3, figsize=(11, 7), sharex=True, sharey=True, layout="constrained")
    for ax, (name, _, _) in zip(axes.flat, group):
        points = initial_samples[name]
        ax.scatter(points[:, 0], points[:, 1], s=22, alpha=0.7, color="#2563a6", edgecolors="none")
        ax.set(title=name, xlim=(-5.3, 5.3), ylim=(-5.3, 5.3), xlabel="$x_1$", ylabel="$x_2$")
        ax.set_aspect("equal")
        ax.set_xticks([-5, 0, 5])
        ax.set_yticks([-5, 0, 5])
        ax.grid(alpha=0.15)
    for ax in list(axes.flat)[len(group) :]:
        ax.set_visible(False)
    fig.suptitle(f"{labels['init_title']} · {INITIAL_AGENTS}")
    initialisation_figures.append(fig)
    plt.show()

initial_summary = []
for name, points in initial_samples.items():
    counts, _, _ = np.histogram2d(points[:, 0], points[:, 1], bins=8, range=[[-5, 5], [-5, 5]])
    initial_summary.append(
        {
            labels["init"]: name,
            labels["unique"]: len(np.unique(points, axis=0)),
            labels["edge"]: 100 * np.any(np.isclose(np.abs(points), 5.0), axis=1).mean(),
            labels["bins"]: np.count_nonzero(counts),
        }
    )
display(pd.DataFrame(initial_summary).round(1))
np.random.seed(SEED)


Los puntos superpuestos parecen uno solo en el gráfico, por eso la tabla cuenta las coordenadas distintas y los puntos sobre los límites. Las celdas ocupadas valen para esta partición de 8 × 8 y no ordenan los métodos. Una muestra concentrada tiene sentido cuando se sabe dónde está el óptimo; sin esa información, una muestra dispersa cubre más del dominio.

## 3. Operadores y colecciones
Un operador es una tupla `(nombre, parámetros, selector)`. El operador propone nuevas posiciones y el selector decide cuáles se quedan: con `greedy`, un individuo solo se mueve si mejora el valor de la función. Los operadores `initialize_*` reinician la población, y la hiperheurística los trata como inicializadores.

Una colección es un archivo de texto con un operador por línea. La colección por defecto se lee del paquete instalado, así que funciona desde cualquier directorio.

In [ ]:
print(labels["operator"], op.__all__)
print(labels["selector"], op.__selectors__)
collection_text = files("customhys").joinpath("collections/default.txt").read_text(encoding="utf-8")
default_operators = [
    ast.literal_eval(line) for line in collection_text.splitlines() if line.strip()
]
print(len(default_operators))
display(
    pd.DataFrame(
        default_operators, columns=[labels["operator"], labels["parameters"], labels["selector"]]
    ).head(10)
)


`obtain_operators` lista las familias de operadores con valores de parámetros, y `build_operators` escribe una colección en `collections/` (relativo al directorio actual). Aquí generamos una colección con dos familias dentro de un directorio temporal, para no sobrescribir nada.

In [ ]:
import os
from tempfile import TemporaryDirectory

operator_families = [
    family
    for family in op.obtain_operators(num_vals=2)
    if family[0] in {"random_search", "spiral_dynamic"}
]
original_dir = Path.cwd()
with TemporaryDirectory() as temporary_dir:
    try:
        os.chdir(temporary_dir)
        op.build_operators(heuristics=operator_families, file_name="tutorial")
        generated_text = Path("collections/tutorial.txt").read_text(encoding="utf-8")
    finally:
        os.chdir(original_dir)
generated_operators = [
    ast.literal_eval(line) for line in generated_text.splitlines() if line.strip()
]
print(len(generated_operators))
display(
    pd.DataFrame(
        generated_operators, columns=[labels["operator"], labels["parameters"], labels["selector"]]
    ).head()
)


Un operador en acción: en cada iteración, `random_search` desplaza a los individuos y el selector `greedy` conserva solo las mejoras. Las líneas grises son los valores de la población y la línea resaltada es el mejor resultado acumulado. Este bucle es, en esencia, lo que la clase `Metaheuristic` automatiza en la siguiente sección.

In [ ]:
operator_name, operator_params, selector = (
    "random_search",
    {"scale": 0.2, "distribution": "uniform"},
    "greedy",
)
history = [pop.fitness.copy()]
best_history = [pop.global_best_fitness]
for iteration in range(1, NUM_ITERATIONS + 1):
    pop.iteration = iteration
    getattr(op, operator_name)(pop, **operator_params)
    pop.evaluate_fitness(fun.get_function_value)
    pop.update_positions("population", selector)
    pop.update_positions("global", "greedy")
    history.append(pop.fitness.copy())
    best_history.append(pop.global_best_fitness)
history = np.asarray(history)
fig, ax = plt.subplots()
ax.plot(history, color="grey", alpha=0.15)
ax.plot(best_history, lw=2, label=labels["best"])
ax.set(xlabel=labels["iteration"], ylabel=labels["fitness"])
ax.legend()
plt.show()
show_positions(pop)


## 4. Metaheurísticas
`Metaheuristic` recibe un problema (`get_formatted_problem()`), una secuencia de operadores y el presupuesto de búsqueda, y aplica la secuencia completa en cada iteración. El ejemplo combina mutación diferencial y cruce en Rastrigin de tres dimensiones, con inicialización `lhs`.

`get_solution()` devuelve coordenadas internas; `rescale_back` las convierte al dominio del problema y permite comprobar el resultado directamente en la función. `historical` registra el estado después de cada operador: con dos operadores, cada iteración genera dos registros, además del inicial.

In [ ]:
problem_fun = bf.Rastrigin(3)
problem = problem_fun.get_formatted_problem()
differential_sequence = [
    (
        "differential_mutation",
        {"expression": "current-to-best", "num_rands": 2, "factor": 0.8},
        "all",
    ),
    ("differential_crossover", {"crossover_rate": 0.2, "version": "binomial"}, "greedy"),
]
met = mh.Metaheuristic(
    problem,
    differential_sequence,
    num_agents=NUM_AGENTS,
    num_iterations=NUM_ITERATIONS,
    initial_scheme="lhs",
)
met.run()
normalised_best, best_fitness = met.get_solution()
best_position = met.pop.rescale_back(normalised_best)
print("x_best =", best_position, "f_best =", best_fitness)
np.testing.assert_allclose(problem_fun.get_function_value(best_position), best_fitness, atol=1e-10)


In [ ]:
def plot_run(metaheuristic):
    recorded = metaheuristic.historical
    positions = metaheuristic.pop.rescale_back(np.asarray(recorded["position"]))
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(recorded["fitness"])
    axes[0].set(xlabel=labels["step"], ylabel=labels["fitness"])
    axes[1].plot(positions)
    axes[1].set(xlabel=labels["step"], ylabel=labels["position"])
    axes[1].legend([f"$x_{i + 1}$" for i in range(metaheuristic.num_dimensions)])
    fig.tight_layout()
    plt.show()


plot_run(met)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(met.historical["centroid"])
axes[0].set(xlabel=labels["step"], ylabel=labels["centroid"])
axes[1].plot(met.historical["radius"])
axes[1].set(xlabel=labels["step"], ylabel=labels["radius"])
fig.tight_layout()
plt.show()


### 4.1 Por dónde pasó la población
`historical` guarda solo el mejor valor y la mejor posición. Para ver toda la población, la subclase siguiente sobrescribe `reset_historicals` y `update_historicals`, que `run()` llama, y guarda todas las posiciones después de cada operador y de la selección.

Los mapas muestran la población conservada tras la selección, así que los movimientos rechazados no aparecen. Las líneas unen registros consecutivos y no indican que los puntos intermedios se hayan evaluado. El mapa de ocupación cuenta visitas repetidas, por eso también refleja cuánto tiempo estuvieron los individuos en cada región.

In [ ]:
from matplotlib.colors import LogNorm


class TracedMetaheuristic(mh.Metaheuristic):
    def reset_historicals(self):
        super().reset_historicals()
        self.population_history = []
        self.population_fitness = []
        self.trace_iterations = []

    def update_historicals(self):
        super().update_historicals()
        self.population_history.append(self.pop.get_positions().copy())
        self.population_fitness.append(self.pop.fitness.copy())
        self.trace_iterations.append(self.pop.iteration)


def plot_exploration(run, objective, boundaries, optimum):
    path = np.asarray(run.population_history)
    lower, upper = np.asarray(boundaries)
    grid_x, grid_y = np.meshgrid(
        np.linspace(lower[0], upper[0], 100), np.linspace(lower[1], upper[1], 100)
    )
    grid_f = np.array(
        [objective([x, y]) for x, y in zip(grid_x.ravel(), grid_y.ravel(), strict=True)]
    )
    grid_f = grid_f.reshape(grid_x.shape)
    best_path = run.pop.rescale_back(np.asarray(run.historical["position"]))
    fig, axes = plt.subplots(2, 3, figsize=(14, 8), layout="constrained")

    def landscape(ax):
        surface = ax.contourf(
            grid_x, grid_y, np.log10(1 + grid_f), levels=22, cmap="Blues", alpha=0.65
        )
        ax.scatter(*optimum, marker="*", s=130, c="#b42318", zorder=5, label=labels["optimum"])
        ax.set(xlim=(lower[0], upper[0]), ylim=(lower[1], upper[1]), xlabel="$x_1$", ylabel="$x_2$")
        return surface

    for ax, index in zip(axes[0], [0, len(path) // 2, len(path) - 1], strict=True):
        surface = landscape(ax)
        ax.scatter(
            path[index, :, 0],
            path[index, :, 1],
            s=22,
            c="#146b5a",
            edgecolors="white",
            linewidths=0.3,
        )
        ax.scatter(*best_path[index], marker="x", s=70, c="#e07800", zorder=6, label=labels["best"])
        ax.set_title(
            f"{labels['iteration']} {run.trace_iterations[index]} · {labels['record']} {index}"
        )
    axes[0, 0].legend(fontsize=8, loc="upper left")
    fig.colorbar(surface, ax=list(axes[0]), label="log₁₀(1 + f)", shrink=0.8)

    landscape(axes[1, 0])
    for agent in range(min(5, run.num_agents)):
        axes[1, 0].plot(path[:, agent, 0], path[:, agent, 1], lw=0.7, alpha=0.45)
    axes[1, 0].plot(best_path[:, 0], best_path[:, 1], color="#e07800", lw=1.8)
    axes[1, 0].set_title(labels["trajectory"])

    visited = path.reshape(-1, 2)
    density = axes[1, 1].hist2d(
        visited[:, 0],
        visited[:, 1],
        bins=25,
        range=list(zip(lower, upper, strict=True)),
        cmap="YlGnBu",
        norm=LogNorm(),
        cmin=1,
    )
    axes[1, 1].scatter(*optimum, marker="*", c="#b42318", s=100)
    axes[1, 1].set(title=labels["occupancy"], xlabel="$x_1$", ylabel="$x_2$")
    fig.colorbar(density[3], ax=axes[1, 1], label=labels["count"], shrink=0.8)

    axes[1, 2].plot(run.historical["fitness"], color="#146b5a", lw=2)
    axes[1, 2].set(
        title=labels["convergence"], xlabel=labels["record"], ylabel="f(x)", yscale="symlog"
    )
    axes[1, 2].set_yscale('symlog', linthresh=1e-6)
    axes[1, 2].set_ylim(bottom=0)
    axes[1, 2].grid(alpha=0.15)
    plt.show()
    return fig


In [ ]:
np.random.seed(SEED)
random.seed(SEED)
visual_fun = bf.Rastrigin(2)
visual_problem = visual_fun.get_formatted_problem()
trace_rastrigin = TracedMetaheuristic(
    visual_problem,
    [("random_search", {"scale": 0.08, "distribution": "gaussian"}, "greedy")],
    num_agents=NUM_AGENTS,
    num_iterations=NUM_ITERATIONS,
    initial_scheme="lhs",
)
trace_rastrigin.run()
rastrigin_figure = plot_exploration(
    trace_rastrigin,
    visual_fun.get_function_value,
    visual_problem["boundaries"],
    optimum=(0, 0),
)


### 4.2 Variación entre ejecuciones
Los algoritmos estocásticos dan resultados distintos en cada ejecución, así que una sola dice poco. Aquí combinamos búsqueda aleatoria, dinámica espiral y búsqueda gravitacional en Griewank de seis dimensiones, cada ejecución con una semilla distinta. La banda entre P25 y P75 contiene la mitad central de las ejecuciones. Usamos solo tres repeticiones para que el ejemplo sea rápido.

In [ ]:
griewank = bf.Griewank(6)
griewank_problem = griewank.get_formatted_problem()
hybrid_sequence = [
    ("random_search", {"scale": 1.0, "distribution": "uniform"}, "greedy"),
    ("spiral_dynamic", {"radius": 0.9, "angle": 33.5, "sigma": 0.1}, "all"),
    ("gravitational_search", {"gravity": 0.5, "alpha": 0.2}, "metropolis"),
]
runs = []
for replica in range(NUM_REPLICAS):
    np.random.seed(SEED + replica)
    random.seed(SEED + replica)
    hybrid_met = mh.Metaheuristic(
        griewank_problem,
        hybrid_sequence,
        num_iterations=NUM_ITERATIONS,
        num_agents=NUM_AGENTS,
    )
    hybrid_met.run()
    runs.append(hybrid_met.historical["fitness"])
fitness_array = np.asarray(runs).T
final_fitness = fitness_array[-1]
display(
    pd.DataFrame(
        {labels["replica"]: np.arange(1, NUM_REPLICAS + 1), labels["fitness"]: final_fitness}
    )
)
plot_run(hybrid_met)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={"width_ratios": [3, 1]})
axes[0].plot(fitness_array, color="grey", alpha=0.4)
axes[0].plot(np.median(fitness_array, axis=1), label="P50", lw=2)
axes[0].fill_between(
    np.arange(len(fitness_array)),
    np.quantile(fitness_array, 0.25, axis=1),
    np.quantile(fitness_array, 0.75, axis=1),
    alpha=0.2,
    label="P25–P75",
)
axes[0].set(xlabel=labels["step"], ylabel=labels["fitness"])
axes[0].legend()
axes[1].boxplot(final_fitness)
axes[1].set(xlabel=labels["replica"], ylabel=labels["fitness"])
fig.tight_layout()
plt.show()


## 5. Hiperheurística
Hasta aquí elegimos el algoritmo a mano. `Hyperheuristic` recibe un espacio de heurísticas (una lista de operadores) y busca una secuencia que funcione bien para el problema. Aquí el problema es Sphere en dos dimensiones, con mínimo cero en `(0, 0)`.

Cada secuencia candidata se ejecuta `num_replicas` veces y se evalúa con `mediana + (P75 − P25)` de los valores finales. La medida combina el resultado típico y su variación; cuanto menor, mejor.

La colección tiene dos inicializadores. El primer operador de la secuencia define cómo empieza la población; un `initialize_*` que aparezca después reinicia la población en medio de la búsqueda. Ese comportamiento debe conservarse al ejecutar la secuencia encontrada.

In [ ]:
heuristic_space = [
    ("initialize_random", {}, "all"),
    ("initialize_lhs", {}, "all"),
    ("random_search", {"scale": 0.2, "distribution": "uniform"}, "greedy"),
    (
        "local_random_walk",
        {"probability": 0.75, "scale": 0.2, "distribution": "gaussian"},
        "greedy",
    ),
]
parameters = {
    "cardinality": 3,
    "cardinality_min": 2,
    "num_iterations": 30,
    "num_agents": NUM_AGENTS,
    "as_mh": True,
    "num_replicas": NUM_REPLICAS,
    "num_steps": HH_STEPS,
    "stagnation_percentage": 0.5,
    "max_temperature": 1.0,
    "min_temperature": 1e-6,
    "cooling_rate": 1e-3,
    "temperature_scheme": "fast",
    "acceptance_scheme": "exponential",
    "allow_weight_matrix": False,
    "trial_overflow": False,
    "learnt_dataset": None,
    "repeat_operators": True,
    "verbose": False,
    "learning_portion": 0.37,
    "solver": "static",
    "initial_scheme": "random",
}
np.random.seed(SEED)
random.seed(SEED)
hh_fun = bf.Sphere(2)
hh_problem = hh_fun.get_formatted_problem()
run_label = "tutorial-" + uuid4().hex
hyp = hh.Hyperheuristic(
    heuristic_space=heuristic_space,
    problem=hh_problem,
    parameters=parameters.copy(),
    file_label=run_label,
)
best_sol, best_perf, hist_current, hist_best = hyp.solve(save_steps=True)
best_sequence = hyp.get_operators(best_sol)
print(labels["performance"], best_perf)
display(
    pd.DataFrame(
        best_sequence, columns=[labels["operator"], labels["parameters"], labels["selector"]]
    )
)


In [ ]:
fig, ax = plt.subplots()
ax.plot(hist_current, label=labels["current"], alpha=0.7)
ax.plot(hist_best, label=labels["best"], lw=2)
ax.set(xlabel=labels["iteration"], ylabel=labels["performance"])
ax.legend()
plt.show()


### 5.1 Leyendo los resultados
Cada ejecución guarda sus archivos en `data_files/raw/<etiqueta>/`. El solver estático registra la solución inicial y las siguientes que igualan o mejoran a la mejor, es decir, no todos los intentos. `merge_json` une los registros en un archivo y `read_json` lo carga.

In [ ]:
results_dir = Path("data_files") / "raw" / run_label
tl.merge_json(results_dir.as_posix())
full_data = tl.read_json(results_dir.with_suffix(".json").as_posix())
print(results_dir.resolve())
display(
    pd.DataFrame(
        [
            {
                "step": record["step"],
                labels["performance"]: record["performance"],
                labels["operator"]: record["encoded_solution"],
            }
            for record in full_data
        ]
    )
)
saved_fitness = full_data[-1]["details"]["fitness"]
fig, ax = plt.subplots()
ax.boxplot(saved_fitness)
ax.set(xlabel=labels["replica"], ylabel=labels["fitness"])
plt.show()


### 5.2 Ejecutando el algoritmo elegido
`get_operators` convierte los índices que devuelve la hiperheurística en la secuencia de operadores. Si el primero es `initialize_*`, pasa a ser el `initial_scheme` y los demás siguen en el mismo orden. La ejecución parte de una población nueva; no continúa ninguna de las repeticiones hechas durante la búsqueda.

In [ ]:
initial_scheme = parameters["initial_scheme"]
discovered_operators = list(best_sequence)
if discovered_operators[0][0].startswith("initialize_"):
    initial_name, initial_params, _ = discovered_operators.pop(0)
    initial_scheme = (initial_name, initial_params)
discovered_met = TracedMetaheuristic(
    hh_problem,
    discovered_operators,
    num_agents=NUM_AGENTS,
    num_iterations=parameters["num_iterations"],
    initial_scheme=initial_scheme,
)
discovered_met.run()
x_normalised, f_best = discovered_met.get_solution()
x_best = discovered_met.pop.rescale_back(x_normalised)
print("x_best =", x_best, "f_best =", f_best)
np.testing.assert_allclose(hh_fun.get_function_value(x_best), f_best, atol=1e-10)
plot_run(discovered_met)


Mapas de una nueva ejecución del algoritmo elegido en Sphere. La hiperheurística busca en el espacio de *secuencias de operadores*; estos mapas muestran la secuencia elegida buscando en el espacio de las *coordenadas del problema*. Son dos espacios distintos.

In [ ]:
sphere_figure = plot_exploration(
    discovered_met, hh_fun.get_function_value, hh_problem["boundaries"], optimum=(0, 0)
)


## 6. La colección CEC2005
`CEC2005` reúne problemas de referencia usados para evaluar algoritmos. F1 en cinco dimensiones es una Sphere desplazada, así que su mínimo no es cero. El uso es el mismo de antes: `get_formatted_problem()` devuelve el diccionario del problema, que va a la metaheurística o a la hiperheurística.

In [ ]:
cec_fun = bf.CEC2005("F1", 5)
cec_problem = cec_fun.get_formatted_problem()
cec_met = mh.Metaheuristic(
    cec_problem,
    differential_sequence,
    num_iterations=NUM_ITERATIONS,
    num_agents=NUM_AGENTS,
)
cec_met.run()
x_normalised, cec_fitness = cec_met.get_solution()
cec_position = cec_met.pop.rescale_back(x_normalised)
print("x_best =", cec_position, "f_best =", cec_fitness)
np.testing.assert_allclose(cec_fun.get_function_value(cec_position), cec_fitness)
plot_run(cec_met)

cec_hyp = hh.Hyperheuristic(
    heuristic_space=heuristic_space,
    problem=cec_problem,
    parameters=parameters.copy(),
)
cec_sol, cec_perf, _, cec_hist_best = cec_hyp.solve(save_steps=False)
print(labels["performance"], cec_perf)
display(
    pd.DataFrame(
        cec_hyp.get_operators(cec_sol),
        columns=[labels["operator"], labels["parameters"], labels["selector"]],
    )
)


## 7. Definiendo un problema propio: Rosenbrock
No hace falta una clase del framework: basta un diccionario. Lleva la función (que recibe el vector de coordenadas), los límites `([xmin, ymin], [xmax, ymax])` y `is_constrained=True` para respetarlos.

Rosenbrock tiene un valle estrecho y curvo:

$$\min_{-2 \le x \le 2,\;-1 \le y \le 3} f(x,y)=(1-x)^2+100(y-x^2)^2.$$

Los dos términos se anulan en **(1, 1)**, donde **f = 0**. Llegar al valle suele ser fácil; recorrerlo hasta el mínimo es lo difícil. Como el óptimo es conocido, podemos comprobar el valor de la función y la distancia hasta él.

In [ ]:
def rosenbrock(point):
    x, y = point
    return float((1 - x) ** 2 + 100 * (y - x**2) ** 2)


rosen_problem = {
    "function": rosenbrock,
    "boundaries": ([-2.0, -1.0], [2.0, 3.0]),
    "is_constrained": True,
}
assert rosenbrock([1.0, 1.0]) == 0.0
assert rosenbrock([0.0, 0.0]) == 1.0

rx, ry = np.meshgrid(np.linspace(-2, 2, 160), np.linspace(-1, 3, 160))
rz = (1 - rx) ** 2 + 100 * (ry - rx**2) ** 2
fig, ax = plt.subplots(figsize=(7, 5), layout="constrained")
surface = ax.contourf(rx, ry, np.log10(1 + rz), levels=25, cmap="Blues")
valley_x = np.linspace(-1, np.sqrt(3), 160)
ax.plot(valley_x, valley_x**2, "--", color="#e07800", lw=1.5, label="$y=x^2$")
ax.scatter(1, 1, s=150, marker="*", color="#b42318", label=labels["optimum"])
ax.set(xlabel="$x$", ylabel="$y$", title="Rosenbrock")
ax.legend()
fig.colorbar(surface, ax=ax, label="log₁₀(1 + f)")
plt.show()


### 7.1 La hiperheurística arma el algoritmo
El espacio de búsqueda tiene movimientos aleatorios en tres escalas, una caminata aleatoria local y muestreo uniforme (`random_sample`). La inicialización queda fija en `lhs`, para que la búsqueda sea solo sobre operadores.

La búsqueda externa prueba hasta 12 cambios de secuencia (`num_steps`). Cada candidato corre 80 iteraciones, con 32 individuos y tres repeticiones. Las secuencias más largas hacen más evaluaciones por iteración. El desempeño mostrado resume las repeticiones; no es el valor de la función en un solo punto.

In [ ]:
rosen_space = [
    ("random_search", {"scale": scale, "distribution": "gaussian"}, "greedy")
    for scale in (0.01, 0.05, 0.2)
] + [
    (
        "local_random_walk",
        {"probability": 0.75, "scale": 0.1, "distribution": "gaussian"},
        "greedy",
    ),
    ("random_sample", {}, "greedy"),
]
rosen_parameters = {
    **parameters,
    "cardinality_min": 1,
    "cardinality": 3,
    "num_iterations": 80,
    "num_steps": 12,
    "stagnation_percentage": 1.0,
    "initial_scheme": "lhs",
}
np.random.seed(SEED)
random.seed(SEED)
rosen_hyp = hh.Hyperheuristic(
    heuristic_space=rosen_space,
    problem=rosen_problem,
    parameters=rosen_parameters.copy(),
)
rosen_sol, rosen_perf, rosen_current, rosen_best = rosen_hyp.solve(save_steps=False)
rosen_sequence = rosen_hyp.get_operators(rosen_sol)
print(labels["performance"], rosen_perf)
display(
    pd.DataFrame(
        rosen_sequence, columns=[labels["operator"], labels["parameters"], labels["selector"]]
    )
)

fig, ax = plt.subplots()
ax.plot(rosen_current, label=labels["current"], alpha=0.6)
ax.plot(rosen_best, label=labels["best"], lw=2)
ax.set(title=labels["search"], xlabel=labels["iteration"], ylabel=labels["performance"])
ax.legend()
plt.show()


### 7.2 Ejecutando y comprobando la solución
Población nueva, semilla nueva. La tabla muestra las coordenadas finales, el valor de la función, la distancia hasta `(1, 1)` y el valor inicial. La función se evalúa de nuevo en esas coordenadas para comprobar el resultado. Con un presupuesto pequeño, puede que no llegue al mínimo exacto.

In [ ]:
np.random.seed(SEED + 1)
random.seed(SEED + 1)
rosen_run = TracedMetaheuristic(
    rosen_problem,
    rosen_sequence,
    num_agents=NUM_AGENTS,
    num_iterations=rosen_parameters["num_iterations"],
    initial_scheme="lhs",
)
rosen_run.run()
rosen_normalised, rosen_fitness = rosen_run.get_solution()
rosen_position = rosen_run.pop.rescale_back(rosen_normalised)
np.testing.assert_allclose(rosenbrock(rosen_position), rosen_fitness, atol=1e-10)
display(
    pd.DataFrame(
        [
            {
                "x": rosen_position[0],
                "y": rosen_position[1],
                "f(x, y)": rosen_fitness,
                labels["distance"]: np.linalg.norm(rosen_position - np.array([1.0, 1.0])),
                labels["initial"]: rosen_run.historical["fitness"][0],
            }
        ]
    )
)
rosen_figure = plot_exploration(rosen_run, rosenbrock, rosen_problem["boundaries"], optimum=(1, 1))


### 7.3 Animación
Reproduce la población ya registrada; no ejecuta una nueva optimización. Se puede pausar y moverse entre cuadros. Los cuadros se muestrean para que el cuaderno no pese (el último siempre se incluye) y no hace falta un codificador de vídeo.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

animation_path = np.asarray(rosen_run.population_history)
animation_best = rosen_run.pop.rescale_back(np.asarray(rosen_run.historical["position"]))
frame_indices = np.unique(np.linspace(0, len(animation_path) - 1, 36, dtype=int))
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), layout="constrained")
axes[0].contourf(rx, ry, np.log10(1 + rz), levels=22, cmap="Blues", alpha=0.65)
axes[0].scatter(1, 1, marker="*", c="#b42318", s=100)
cloud = axes[0].scatter([], [], c="#146b5a", s=18)
marker = axes[0].scatter([], [], marker="x", c="#e07800", s=60)
axes[0].set(xlim=(-2, 2), ylim=(-1, 3), xlabel="$x$", ylabel="$y$")
axes[1].plot(rosen_run.historical["fitness"], color="#146b5a")
cursor = axes[1].axvline(0, color="#e07800")
axes[1].set(xlabel=labels["record"], ylabel="f(x)", title=labels["convergence"], yscale="symlog")


axes[1].set_yscale('symlog', linthresh=1e-6)
axes[1].set_ylim(bottom=0)

def update_frame(index):
    cloud.set_offsets(animation_path[index])
    marker.set_offsets(animation_best[index : index + 1])
    cursor.set_xdata([index, index])
    axes[0].set_title(f"{labels['iteration']} {rosen_run.trace_iterations[index]}")
    return cloud, marker, cursor


animation = FuncAnimation(fig, update_frame, frames=frame_indices, interval=180, repeat=False)
animation_html = animation.to_jshtml(default_mode="once")
plt.close(fig)
display(HTML(animation_html))


## Para experimentar
- Cambie solo `initial_scheme`, manteniendo la función, los operadores y el número de evaluaciones, y compare el valor final y la región ocupada.
- En Rosenbrock, `num_steps` prueba más algoritmos y `num_iterations` da más tiempo a cada uno.
- [quickTest](quickTest.ipynb) usa el operador de sistemas lineales.